# Example 05 · Human-in-the-Loop (HITL)

**Source:** [LangChain Docs — Human-in-the-Loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)

---

## Objectives

By the end of this notebook you will be able to:

1. Understand **why** human-in-the-loop (HITL) oversight matters for AI agents
2. Use LangGraph's native `interrupt()` to pause and resume graph execution
3. Attach `HumanInTheLoopMiddleware` to an agent for tool-call approvals
4. Handle all four decision types: **approve**, **edit**, **reject**, **respond**
5. Write **conditional interrupt** rules that only pause on risky actions

---

## Background

### Why Human Oversight?

Autonomous agents can take irreversible actions — deleting records, writing files,
sending emails. A **human-in-the-loop** checkpoint lets a person review, modify, or
cancel proposed actions before they execute.

```
  Agent plans action
        │
        ▼
  ┌──────────────┐
  │  HITL Check  │  ◄─── is this tool risky?
  └──────┬───────┘
         │
    ┌────┴────┐
    │  Human  │  approve / edit / reject / respond
    └────┬────┘
         │
        ▼
  Tool executes (or is cancelled)
```

### Two HITL Patterns in LangChain

| Pattern | When to use |
|---------|-------------|
| **LangGraph `interrupt()`** | You control the graph; need fine-grained control over pause points |
| **`HumanInTheLoopMiddleware`** | You use `create_agent`; need policy-based tool approval |

### Decision Types

| Decision | Effect |
|----------|--------|
| **approve** | Run the tool call exactly as proposed |
| **edit** | Modify arguments before running |
| **reject** | Cancel the call and send feedback to the agent |
| **respond** | Provide a human reply as the tool result ("ask-user" pattern) |

Run cells **top to bottom**.

## Setup

All imports in one cell — re-run to reset state.

In [1]:
import sys
from pathlib import Path

# Add project root so the shared `common/` package is importable
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# LangGraph primitives
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

from common.env import get_env   # loads .env
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

handler = create_langfuse_handler()

def make_config(trace_name: str, thread_id: str = "s05") -> dict:
    cfg = build_langfuse_config(handler, session_id="s05-hitl", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ Setup complete")

✓ Setup complete


---

## Part 1 · LangGraph Native `interrupt()`

LangGraph's `interrupt()` function is the foundation of all HITL patterns.
When called inside a graph node, it:

1. **Pauses** the graph at that node
2. **Saves** the entire graph state to the checkpointer
3. **Returns** an `Interrupt` object to the caller

The graph stays paused until you send a `Command(resume=...)` to wake it up.

```
graph.invoke(input)      # runs until interrupt()
      │
      ▼  paused — state saved in checkpointer
graph.invoke(Command(resume=answer))  # continues from where it stopped
```

### Step 1a — Build a simple approval graph

In [2]:
# Define graph state
class ReviewState(TypedDict):
    action: str          # the action the agent wants to take
    approved: bool       # whether it was approved
    result: str          # what happened

def plan_node(state: ReviewState) -> ReviewState:
    """Agent plans an action (simulate with a fixed string)."""
    print(f"🤖 Agent wants to: {state['action']}")
    return state

def approval_node(state: ReviewState) -> ReviewState:
    """Pause here and wait for human approval."""
    # interrupt() pauses the graph and sends the value to the caller
    human_decision = interrupt({
        "question": f"Approve this action?",
        "action": state["action"],
    })
    # Execution resumes here after Command(resume=...) is sent
    return {"approved": human_decision == "yes"}

def execute_node(state: ReviewState) -> ReviewState:
    """Run or skip the action based on the approval decision."""
    if state["approved"]:
        result = f"✅ Executed: {state['action']}"
    else:
        result = f"❌ Cancelled: {state['action']}"
    print(result)
    return {"result": result}

# Build the graph with an in-memory checkpointer (required for interrupt)
checkpointer = InMemorySaver()

builder = StateGraph(ReviewState)
builder.add_node("plan", plan_node)
builder.add_node("approval", approval_node)
builder.add_node("execute", execute_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "approval")
builder.add_edge("approval", "execute")
builder.add_edge("execute", END)

review_graph = builder.compile(checkpointer=checkpointer)

print("✓ Graph compiled")

✓ Graph compiled


### Step 1b — Run until the interrupt fires

In [3]:
# Use a unique thread_id so state is stored per conversation
cfg = {"configurable": {"thread_id": "hitl-demo-1"}}

# First call — runs plan_node, hits interrupt() in approval_node, and stops
result = review_graph.invoke(
    {"action": "DELETE all records older than 30 days", "approved": False, "result": ""},
    config=cfg,
)

# result.interrupts  → list of Interrupt objects (LangGraph v1.1+)
# result.value       → the graph state dict
print("Graph paused. Interrupt payload:")
print(result.interrupts)

### Step 1c — Resume with an approval decision

In [4]:
# Show what the graph is waiting for, then ask the human
pending = result.interrupts[0].value
print(f"\n\u26a0\ufe0f  Pending approval for: {pending['action']}")
print(f"   {pending['question']}")

# input() blocks until the user types something and presses Enter
decision = input("\nYour decision (yes / no): ").strip().lower()

# Resume the graph — continues from where interrupt() paused it
final = review_graph.invoke(
    Command(resume=decision),
    config=cfg,
)
print("Final state:", final.value["result"])

In [5]:
# Reject: start a new thread and reject the same action
cfg2 = {"configurable": {"thread_id": "hitl-demo-2"}}

review_graph.invoke(
    {"action": "DROP TABLE users", "approved": False, "result": ""},
    config=cfg2,
)

final2 = review_graph.invoke(
    Command(resume="no"),   # human says no
    config=cfg2,
)
print("Final state:", final2.value["result"])

---

## Part 2 · `HumanInTheLoopMiddleware` with `create_agent`

`HumanInTheLoopMiddleware` is a higher-level abstraction that sits between the LLM
and the tools. It intercepts tool calls, checks a policy, and pauses the agent if
human approval is needed — all without you building a custom graph.

```
LLM decides to call write_file()
        │
        ▼
  ┌──────────────────────────┐
  │ HumanInTheLoopMiddleware │  interrupt_on={"write_file": True}
  └──────────┬───────────────┘
             │  interrupt fired
             ▼
  Human sends Command(resume={"decisions": [{"type": "approve"}]})
             │
             ▼
       write_file() runs
```

### Step 2a — Define tools and create the agent

In [6]:
# Simulate risky tools (no real side effects)
@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file at the given path."""
    return f"[simulated] Wrote {len(content)} bytes to {path}"

@tool
def execute_sql(query: str) -> str:
    """Execute a SQL query against the database."""
    return f"[simulated] Ran SQL: {query}"

@tool
def read_data(table: str) -> str:
    """Read rows from a database table (read-only, safe)."""
    return f"[simulated] Read 42 rows from '{table}'"

# HumanInTheLoopMiddleware configuration:
#   True  → always interrupt when this tool is called
#   False → never interrupt (run freely)
agent = create_agent(
    model=create_llm(),
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file":  True,   # always ask before writing
                "execute_sql": True,   # always ask before SQL
                "read_data":   False,  # safe to run without approval
            },
        )
    ],
    checkpointer=InMemorySaver(),   # required for interrupt/resume
)

print("✓ Agent created with HITL middleware")

✓ Agent created with HITL middleware


### Step 2b — Trigger an interrupt (agent proposes a risky action)

In [7]:
hitl_cfg = make_config("HITL Middleware Demo", thread_id="s05-hitl-1")

# The agent will decide to call write_file — which triggers the middleware
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Save a summary report to /reports/summary.txt"}]},
    config=hitl_cfg,
    version="v2",
)

print("Interrupt payload:")
for intr in result.interrupts:
    # Print the raw interrupt value so you can see its actual structure
    print(f"  {intr.value}")

Interrupt payload:
  {'action_requests': [{'name': 'write_file', 'args': {'path': '/reports/summary.txt', 'content': 'Summary Report\n\nThis report provides an overview of the key findings and insights gathered from the recent analysis. The main points include:\n\n1. **Key Findings**: \n   - Finding 1: Description of finding 1.\n   - Finding 2: Description of finding 2.\n   - Finding 3: Description of finding 3.\n\n2. **Insights**: \n   - Insight 1: Description of insight 1.\n   - Insight 2: Description of insight 2.\n\n3. **Recommendations**: \n   - Recommendation 1: Description of recommendation 1.\n   - Recommendation 2: Description of recommendation 2.\n\nThis report aims to provide actionable insights and guide future decisions.'}, 'description': "Tool execution requires approval\n\nTool: write_file\nArgs: {'path': '/reports/summary.txt', 'content': 'Summary Report\\n\\nThis report provides an overview of the key findings and insights gathered from the recent analysis. The main po

### Step 2c — Decision: **Approve**

Resume with `{"type": "approve"}` to execute the tool exactly as proposed.

In [16]:
approved = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=hitl_cfg,
    version="v2",
)
# .value is the output state dict
approved.value["messages"][-1].pretty_print()

### Step 2d — Decision: **Reject**

Resume with `{"type": "reject"}` plus a `message` explaining why. The agent receives
the feedback and can try a safer approach.

In [9]:
# Start a fresh thread for the reject demo
reject_cfg = make_config("HITL Reject Demo", thread_id="s05-hitl-2")

# Trigger the middleware again
agent.invoke(
    {"messages": [{"role": "user", "content": "Delete old records from the orders table"}]},
    config=reject_cfg,
    version="v2",
)

# Reject: the agent gets the message as tool feedback and replans
rejected = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Too risky. Archive instead of delete."}]}),
    config=reject_cfg,
    version="v2",
)
rejected.value["messages"][-1].pretty_print()

### Step 2e — Decision: **Edit**

Resume with `{"type": "edit"}` and supply modified arguments. Use this when the tool
call is almost right but needs a small correction (e.g., different path or safer query).

In [10]:
edit_cfg = make_config("HITL Edit Demo", thread_id="s05-hitl-3")

# Agent proposes writing to /etc/config (risky path)
agent.invoke(
    {"messages": [{"role": "user", "content": "Write the config to /etc/config.json"}]},
    config=edit_cfg,
    version="v2",
)

# Edit: redirect write to a safer path
edited = agent.invoke(
    Command(resume={"decisions": [{
        "type": "edit",
        "edited_action": {
            "name": "write_file",
            "args": {"path": "/workspace/config.json", "content": "{}"}
        }
    }]}),
    config=edit_cfg,
    version="v2",
)
edited.value["messages"][-1].pretty_print()

### Step 2f — Decision: **Respond**

Resume with `{"type": "respond"}` to provide a human-supplied answer as the tool result.
This implements the **"ask-user"** pattern — the tool call becomes a conversation turn.

In [11]:
respond_cfg = make_config("HITL Respond Demo", thread_id="s05-hitl-4")

@tool
def ask_user(question: str) -> str:
    """Ask the human a clarifying question and return their reply."""
    return "(this is always intercepted by the middleware)"

ask_agent = create_agent(
    model=create_llm(),
    tools=[ask_user],
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"ask_user": True})
    ],
    checkpointer=InMemorySaver(),
)

ask_agent.invoke(
    {"messages": [{"role": "user", "content": "What color should the button be?"}]},
    config=respond_cfg,
    version="v2",
)

# Respond: provide the human's answer as the tool result
response = ask_agent.invoke(
    Command(resume={"decisions": [{"type": "respond", "message": "Blue."}]}),
    config=respond_cfg,
    version="v2",
)
response.value["messages"][-1].pretty_print()

---

## Part 3 · Conditional Interrupts

Always interrupting is safe but noisy. Use a **`when`** predicate to only pause
when the tool call meets certain criteria (e.g., writes outside the workspace).

The `when` function receives a `ToolCallRequest` and returns `True` (interrupt) or
`False` (run freely).

### Step 3a — Interrupt only for writes outside `/workspace/`

In [12]:
def writes_outside_workspace(request) -> bool:
    """Return True only when the target path is outside /workspace/."""
    path = request.tool_call["args"].get("path", "")
    return not path.startswith("/workspace/")

conditional_agent = create_agent(
    model=create_llm(),
    tools=[write_file, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "when": writes_outside_workspace,   # conditional!
                },
                "read_data": False,
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

print("✓ Conditional agent created")

✓ Conditional agent created


In [13]:
# Case A: write inside /workspace/ — no interrupt, runs immediately
safe_cfg = make_config("Conditional HITL \u2014 safe", thread_id="s05-cond-1")

safe_result = conditional_agent.invoke(
    {"messages": [{"role": "user", "content": "Write a log to /workspace/logs/run.log"}]},
    config=safe_cfg,
    version="v2",
)
print("No interrupt \u2014 runs straight through:")
safe_result.value["messages"][-1].pretty_print()

In [14]:
# Case B: write outside /workspace/ — interrupt fires
risky_cfg = make_config("Conditional HITL \u2014 risky", thread_id="s05-cond-2")

risky_result = conditional_agent.invoke(
    {"messages": [{"role": "user", "content": "Write credentials to /etc/secrets.txt"}]},
    config=risky_cfg,
    version="v2",
)
print("Interrupt fired for risky write:")
for intr in risky_result.interrupts:
    print(f"  {intr.value}")

# Reject the risky write
final = conditional_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Never write to /etc."}]}),
    config=risky_cfg,
    version="v2",
)
final.value["messages"][-1].pretty_print()

---

## Summary

| Concept | Key API | When to use |
|---------|---------|-------------|
| Native interrupt | `interrupt(value)` in a graph node | Custom graphs with fine-grained control |
| Resume execution | `Command(resume=answer)` | Same thread_id as the paused graph |
| Middleware approval | `HumanInTheLoopMiddleware(interrupt_on={...})` | `create_agent` with policy-based approval |
| Approve | `{"type": "approve"}` | Run tool exactly as proposed |
| Reject | `{"type": "reject", "message": "..."}` | Cancel with feedback for the agent |
| Edit | `{"type": "edit", "edited_action": {...}}` | Modify args before running |
| Respond | `{"type": "respond", "message": "..."}` | Human answers directly (ask-user pattern) |
| Conditional | `"when": callable` | Interrupt only when predicate returns True |

### Critical Requirements

1. **Checkpointer is mandatory** — `InMemorySaver` for dev, `AsyncPostgresSaver` for production
2. **Thread ID must be stable** — same `thread_id` in the config for both the initial call and the resume
3. **Use `version="v2"` or `version="v3"`** when calling `invoke` or `stream_events` with middleware

```
Traces: run make_config() and check Langfuse
```

In [15]:
print(f"Traces available at: {get_langfuse_host()}")

Traces available at: http://localhost:3000
